In [ ]:
"""
05_velorama_stability.py
Is the Velorama GRN seed-robust, and under what regime? One investigation with
five sections that together fix and certify the stable configuration used by
steps 04 and 07:

  A  curate a regulator universe (the keystone: cap predictors so n >> p)
  B  convergence check (training stops on convergence, not at max_iter)
  C  lam sweep -> CHOSEN_LAM (most reproducible top edges)
  D  MAIN run: full cells, curated regs, chosen lam -> stability verdict + consensus
  E  ablation at 2,000 cells -> shows instability was a cell-count (n<p) problem
  F  raw vs noisyR-cleaned universe -> shows whether denoising helps (it didn't)

Reads : liver_endstage.h5ad (step 04), noisyr_signal_genes.txt (step 02)
Writes: 99_results/velorama/seed_stability/*  (consensus GRN, metrics, plots)
"""
import os, time, glob
from gribben_config import *          # paths, gene sets, LAG, CAP_REGULATORS, CHOSEN_LAM, SIGNAL_GENES_TXT
import numpy as np, pandas as pd, scipy.sparse as sp, torch
import scanpy as sc, anndata as ad, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from joblib import Parallel, delayed
from velorama import train_model
from velorama.utils import construct_dag, calculate_diffusion_lags
for v in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ[v] = "1"

RUN_DIR = os.path.join(VELO_DIR, "seed_stability"); os.makedirs(RUN_DIR, exist_ok=True)
SEEDS, N_TARGETS, TOP_PCT, SUBSAMPLE_SEED = [42, 43, 44, 45, 46], 40, 5, 7
WORKERS = max(1, min(4, N_CPUS))
# experimental base config (lam/seed set per run). This is the convergence-aware
# regime; the winning values become STABLE_TRAIN_CONFIG in gribben_config.
BASE = dict(reg_target=False, lr=0.005, lam_ridge=1e-4, penalty="H", lag=LAG,
            hidden=[16], max_iter=1000, device="cpu", lookback=20, check_every=25,
            verbose=False, dynamics="pseudotime")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100


In [ ]:
# === load curated AnnData + recover gene roles ===
adata = ad.read_h5ad(CURATED_H5AD)
if "velorama_reg_names" in adata.uns:
    adata.var["is_reg"]    = adata.var_names.isin(adata.uns["velorama_reg_names"])
    adata.var["is_target"] = adata.var_names.isin(adata.uns["velorama_target_names"])

In [ ]:
# === SECTION A: curated regulator universe ===============================
# Keep lineage TFs + Flufftail hubs, fill the rest by variance up to CAP_REGULATORS.
# Fewer, informative predictors -> well-conditioned per-target regressions.
X0 = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
gene_var  = pd.Series(np.nan_to_num(X0.var(axis=0)), index=adata.var_names)
must_keep = (set(HEPATO_TFs) | set(CHOLANGIO_TFs) | set(FLUFFTAIL_HUBS)) & set(adata.var_names)
reg_pool  = [g for g in adata.var_names[adata.var.is_reg.values] if g not in must_keep]
curated_regs = sorted(must_keep | set(gene_var[reg_pool].sort_values(ascending=False)
                                      .index[:max(0, CAP_REGULATORS - len(must_keep))]))
print(f"Curated regulators: {len(curated_regs)} (must-keep {len(must_keep)})")

# ---- shared helpers -----------------------------------------------------
def build_inputs(cell_idx, universe):
    """Scale a cell/gene slice, build the pseudotime DAG, return Velorama tensors."""
    a = adata[cell_idx][:, universe].copy()
    sc.pp.scale(a)
    a.uns["iroot"] = int(np.argmin(a.obs["dpt_pseudotime"].values))
    A = construct_dag(a, dynamics="pseudotime", ptloc=None, proba=False)
    Xn = np.ascontiguousarray(np.nan_to_num(
        a.X.toarray() if sp.issparse(a.X) else np.asarray(a.X)).astype(np.float32))
    AX = calculate_diffusion_lags(A, torch.tensor(Xn), LAG)
    AXn = AX.numpy() if isinstance(AX, torch.Tensor) else np.asarray(AX, dtype=np.float32)
    reg_names = [g for g in a.var_names if g in set(curated_regs)]
    return list(a.var_names), reg_names, Xn, AXn, a.n_obs

def _train(pos, gene, X, AXn, cfg, rdir, dname):
    import torch; from velorama import train_model
    torch.set_num_threads(1)
    AXt = torch.tensor(AXn, dtype=torch.float32)
    train_model({"AX": AXt, "AY": AXt, "Y": torch.tensor(X[:, [pos]], dtype=torch.float32),
                 "name": gene, "results_dir": rdir, "dir_name": dname, **cfg})

def _load(pt_dir, gene):
    hit = [p for p in glob.glob(os.path.join(pt_dir, f"{gene}.*lag{LAG}.pseudotime.pt"))
           if not p.endswith(".ignore_lag.pt")]
    if not hit: return None
    GC = torch.load(hit[0], map_location="cpu").detach().numpy()[0]
    return GC.max(axis=1) if GC.ndim == 2 else GC

def run_seed(seed, targets, var_names, reg_names, X, AXn, lam, tag):
    """Train all targets for one seed, return the regulators x targets GRN."""
    dname = f"{tag}_lam{lam}_s{seed}"; pt_dir = os.path.join(RUN_DIR, dname)
    pos = {g: i for i, g in enumerate(var_names)}
    todo = [g for g in targets if _load(pt_dir, g) is None]
    if todo:
        Parallel(n_jobs=WORKERS, backend="loky")(
            delayed(_train)(pos[g], g, X, AXn, {**BASE, "lam": lam, "seed": seed}, RUN_DIR, dname)
            for g in todo)
    reg_pos = [pos[r] for r in reg_names]
    mat = pd.DataFrame(0.0, index=reg_names, columns=targets)
    for g in targets:
        full = _load(pt_dir, g)
        if full is not None: mat[g] = np.asarray(full)[reg_pos]
    return mat

def stability(grn_per_seed):
    """Cross-seed reproducibility: correlation of scores + Jaccard of top-5% edges."""
    seeds = list(grn_per_seed); rows = []
    def topset(df):
        v = df.values.flatten(); thr = np.percentile(v, 100 - TOP_PCT)
        r, c = np.where(df.values > thr); return set(zip(r, c))
    for i, s1 in enumerate(seeds):
        for s2 in seeds[i+1:]:
            a, b = grn_per_seed[s1].values.flatten(), grn_per_seed[s2].values.flatten()
            A, B = topset(grn_per_seed[s1]), topset(grn_per_seed[s2])
            rows.append(dict(pearson=pearsonr(a, b)[0],
                             jaccard=len(A & B) / len(A | B) if (A | B) else 1.0))
    d = pd.DataFrame(rows)
    return dict(pearson=d.pearson.mean(), jaccard=d.jaccard.mean(), jaccard_min=d.jaccard.min())

In [ ]:
# --- VIZ: curated regulator universe composition ---
plt.figure(figsize=(4,3.5))
plt.bar(['must-keep\n(lineage+hubs)','variance-filled'],
        [len(must_keep), len(curated_regs)-len(must_keep)], color=['#e88035','#3576b0'])
plt.title(f'Curated regulators: {len(curated_regs)} total'); plt.tight_layout(); plt.show()


In [ ]:
# === build the shared full-cell inputs (targets drawn once, reused everywhere) ===
rng = np.random.default_rng(SUBSAMPLE_SEED)
tgt_pool = [g for g in adata.var_names[adata.var.is_target.values] if g not in curated_regs]
TARGETS  = sorted(rng.choice(tgt_pool, min(N_TARGETS, len(tgt_pool)), replace=False))
UNIVERSE = sorted(set(curated_regs) | set(TARGETS))
VAR, REG, X_NP, AX_NP, n_all = build_inputs(np.arange(adata.n_obs), UNIVERSE)
print(f"Full-cell universe: {len(VAR)} genes | n/p = {n_all/len(VAR):.1f} (want >>1)")

In [ ]:
# === SECTION B: convergence check (one gene, verbose) ====================
pos0 = VAR.index(TARGETS[0])
train_model({"AX": torch.tensor(AX_NP), "AY": torch.tensor(AX_NP),
             "Y": torch.tensor(X_NP[:, [pos0]], dtype=torch.float32), "name": TARGETS[0],
             "results_dir": RUN_DIR, "dir_name": "convcheck",
             **{**BASE, "lam": CHOSEN_LAM, "seed": 42, "verbose": True}})

In [ ]:
# === SECTION C: lam sweep -> most seed-stable sparsity ===================
sweep = []
for lam in [0.02, 0.05, 0.1]:
    gps = {s: run_seed(s, TARGETS[:15], VAR, REG, X_NP, AX_NP, lam, "sweep") for s in SEEDS[:3]}
    st = stability(gps); st["lam"] = lam; sweep.append(st)
    print(f"lam={lam}: pearson={st['pearson']:.3f} jaccard={st['jaccard']:.3f}")
CHOSEN = float(pd.DataFrame(sweep).set_index("lam").jaccard.idxmax())
print("CHOSEN_LAM (this run) =", CHOSEN, "| config default =", CHOSEN_LAM)

In [ ]:
# --- VIZ: lam sweep -- stability metrics vs lam ---
sw = pd.DataFrame(sweep).set_index('lam')
fig, ax = plt.subplots(figsize=(6,4))
sw[['pearson','jaccard']].plot(marker='o', ax=ax)
ax.axvline(CHOSEN, color='crimson', linestyle='--', label=f'chosen lam={CHOSEN}')
ax.set_title('Lam sweep: seed-stability metrics'); ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# === SECTION D: main run + verdict + consensus GRN =======================
grn_per_seed = {}
for s in SEEDS:
    grn_per_seed[s] = run_seed(s, TARGETS, VAR, REG, X_NP, AX_NP, CHOSEN, "main")
st = stability(grn_per_seed)
verdict = ("STABLE" if st["pearson"] >= 0.95 and st["jaccard_min"] >= 0.6 else
           "ACCEPTABLE" if st["pearson"] >= 0.85 else "SEED-SENSITIVE")
print(f"MAIN: pearson={st['pearson']:.3f} jaccard={st['jaccard']:.3f} -> {verdict}")

stack = np.stack([grn_per_seed[s].values for s in SEEDS])       # consensus over seeds
mean_grn = pd.DataFrame(stack.mean(0), index=REG, columns=TARGETS)
std_grn  = pd.DataFrame(stack.std(0),  index=REG, columns=TARGETS)
edges = mean_grn.stack().reset_index(); edges.columns = ["TF", "target", "mean_score"]
edges["std"] = std_grn.stack().values
edges["cv"]  = edges["std"] / (edges.mean_score.abs() + 1e-9)   # per-edge seed variability
edges = edges[edges.TF != edges.target].sort_values("mean_score", ascending=False)
edges.to_csv(os.path.join(RUN_DIR, "consensus_edges_ranked.csv"), index=False)
print(f"Robust edges (top-10% strength & CV<0.5): "
      f"{len(edges[(edges.mean_score > edges.mean_score.quantile(0.9)) & (edges.cv < 0.5)])}")

In [ ]:
# --- VIZ: main-run verdict bars + consensus edge heatmap ---
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].bar(['pearson','jaccard','jaccard_min'],
          [st['pearson'], st['jaccard'], st['jaccard_min']], color='#3576b0')
ax[0].axhline(0.95, color='green', linestyle='--', alpha=.6)
ax[0].set_ylim(0,1.05); ax[0].set_title(f'Seed stability -> {verdict}')
top = edges.head(15).iloc[::-1]
ax[1].barh([f'{r.TF}->{r.target}' for _, r in top.iterrows()], top.mean_score,
           xerr=top['std'], color='#3576b0')
ax[1].set_title('Top consensus edges (±std across seeds)'); ax[1].tick_params(labelsize=7)
plt.tight_layout(); plt.show()


In [ ]:
# === SECTION E: 2,000-cell ablation (was the instability just n<p?) =======
cells2k = np.sort(rng.choice(adata.n_obs, 2000, replace=False))
VARa, REGa, Xa, AXa, _ = build_inputs(cells2k, UNIVERSE)
gps2k = {s: run_seed(s, TARGETS, VARa, REGa, Xa, AXa, CHOSEN, "ablation2k") for s in SEEDS}
sa = stability(gps2k)
print(f"ABLATION 2k cells: pearson={sa['pearson']:.3f} jaccard={sa['jaccard']:.3f} "
      f"(compare to full-cell {st['pearson']:.3f}/{st['jaccard']:.3f})")

In [ ]:
# --- VIZ: full-cell vs 2k-cell ablation comparison ---
plt.figure(figsize=(5,3.5))
x = np.arange(2); w = 0.35
plt.bar(x-w/2, [st['pearson'], st['jaccard']], w, label='full cells', color='#3576b0')
plt.bar(x+w/2, [sa['pearson'], sa['jaccard']], w, label='2k cells', color='#e88035')
plt.xticks(x, ['pearson','jaccard']); plt.legend()
plt.title('Stability: full cells vs 2k-cell ablation'); plt.tight_layout(); plt.show()


In [ ]:
# === SECTION F: raw vs noisyR-cleaned universe ============================
if os.path.exists(SIGNAL_GENES_TXT):
    signal = set(open(SIGNAL_GENES_TXT).read().split()) & set(adata.var_names)
    clean_universe = sorted(set(UNIVERSE) & signal)             # regs+targets that survive cleaning
    VARc, REGc, Xc, AXc, _ = build_inputs(np.arange(adata.n_obs), clean_universe)
    clean_targets = [t for t in TARGETS if t in set(VARc)]
    gpsc = {s: run_seed(s, clean_targets, VARc, REGc, Xc, AXc, CHOSEN, "noisyr_clean") for s in SEEDS}
    sc_ = stability(gpsc)
    print(f"noisyR-CLEANED: pearson={sc_['pearson']:.3f} jaccard={sc_['jaccard']:.3f} "
          f"(vs raw {st['pearson']:.3f}/{st['jaccard']:.3f}) "
          f"-> {'helps' if sc_['jaccard'] > st['jaccard'] + 0.02 else 'no improvement'}")
else:
    print("noisyR signal list missing (run 02_noisyr_clean.R); skipping section F.")

## NEXT STEPS / GAPS
## - Feed the winning CHOSEN back into gribben_config.CHOSEN_LAM, then re-run step 04/07.
## - Sections run in one process; for a big sweep, cache .pt dirs (already keyed by
##   tag/lam/seed) and parallelise across seeds.
## - Stability is measured on N_TARGETS=40 sampled targets; widen for a final claim.
## - construct_dag reads dpt orientation; if you switch the DAG to Monocle3 pseudotime
##   (step 07 option), re-certify stability here under that ordering.

In [ ]:
# --- VIZ: raw vs noisyR-cleaned stability comparison ---
if os.path.exists(SIGNAL_GENES_TXT):
    plt.figure(figsize=(5,3.5))
    x = np.arange(2); w = 0.35
    plt.bar(x-w/2, [st['pearson'], st['jaccard']], w, label='raw', color='#3576b0')
    plt.bar(x+w/2, [sc_['pearson'], sc_['jaccard']], w, label='noisyR-cleaned', color='#4C9BE8')
    plt.xticks(x, ['pearson','jaccard']); plt.legend()
    plt.title('Stability: raw vs noisyR-cleaned universe'); plt.tight_layout(); plt.show()
